<a href="https://colab.research.google.com/github/ArlLps/Facom_LLMs/blob/main/GSI073_aula0_luong_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparação dos dados

Esta tarefa é inverter sequências de caracteres. Exemplo: **aabcd** em **dcbaa**.


In [ ]:
import torch
import torch.nn as nn
import random
import torch.nn.functional as F

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dados
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(50000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Veja um par

In [ ]:
print(pairs[1])

# Definição do modelo Seq2Seq com GRU

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        outputs, h = self.gru(x)
        return outputs, h   # <--- ESSENCIAL

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (B, 1, H)
        encoder_outputs: (B, S, H)

        Retorna:
          context: (B, 1, H)
          attn_weights: (B, 1, S)
        """

        # score = h_t · h_s^T
        # (B, 1, H) x (B, H, S) -> (B, 1, S)
        attn_scores = torch.bmm(decoder_hidden, encoder_outputs.transpose(1, 2))

        attn_weights = F.softmax(attn_scores, dim=-1)  # normaliza nos steps da source

        # context = soma ponderada
        # (B, 1, S) x (B, S, H) -> (B, 1, H)
        context = torch.bmm(attn_weights, encoder_outputs)

        return context, attn_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.attn = LuongAttention()

        # Luong concat: concatena hidden + context
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, h, encoder_outputs):
        """
        x: tokens anteriores corretos  (B, T)
        h: estado inicial do decoder   (1, B, H)
        encoder_outputs: todos os h_s  (B, S, H)
        """
        x = self.embed(x)  # (B, T, E)

        outputs = []
        seq_len = x.size(1)
        hidden = h

        for t in range(seq_len):
            inp = x[:, t:t+1]  # (B, 1, E)

            out_t, hidden = self.gru(inp, hidden)   # out_t: (B,1,H)

            # Atenção
            context, attn_w = self.attn(out_t, encoder_outputs)

            # concatenação [out_t ; context]
            combined = torch.cat([out_t, context], dim=-1)

            logits = self.fc(combined)  # (B,1,V)
            outputs.append(logits)

        outputs = torch.cat(outputs, dim=1)  # (B, T, V)
        return outputs, hidden


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        encoder_outputs, h = self.encoder(src)
        logits, _ = self.decoder(tgt[:, :-1], h, encoder_outputs)
        return logits

# Código para usar o modelo treinado: inferência

In [ ]:
def decode_step(decoder, token, h, encoder_outputs):
    """
    Executa um passo de decodificação:
    - token: tensor (B,1)
    - h: estado oculto do decoder (1,B,H)
    - encoder_outputs: (B,S,H)
    """
    logits, h = decoder(token, h, encoder_outputs)  # (B,1,V)
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)  # (B,1)
    return next_token, h


def predict(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        # codifica entrada
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        # encoder agora retorna (encoder_outputs, h)
        encoder_outputs, h = model.encoder(src)

        # token inicial (ex: espaço ou <sos>)
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)

        seq_invertida = []
        for _ in range(max_len):
            token, h = decode_step(model.decoder, token, h, encoder_outputs)
            seq_invertida.append(token.item())

        return decode(seq_invertida)


# Preparação para treino

In [ ]:
emb_size = 32
hidden_size = 64
encoder = Encoder(vocab_size, emb_size, hidden_size)
decoder = Decoder(vocab_size, emb_size, hidden_size)
model = Seq2Seq(encoder, decoder).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Execução do treino

In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt.zero_grad()
        logits = model(xb, yb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")

# Vamos testar

In [ ]:
acertos = []

In [ ]:
for __ in range(100):
  count = 0

  for _ in range(100):
      s = random_seq()
      pred = predict(model, s, max_len=len(s))
      count += s == pred[::-1]
      # print(f"{s} -> {pred}")
  acertos.append(count)

print(f'''Última Saída: {count}%\nMaior quantidade de acertos: {max(acertos)}%\nMenor quantidade de acertos: {min(acertos)}%\nTentativas: {len(acertos)}''')

# Exercício
Compare o resultado do uso do encoder-decoder com atenção com o encoder-decoder sem atenção.

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parâmetros treináveis: {total_params}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def decode_step_with_attention(decoder, token, h, encoder_outputs):
    """
    Executa um passo de decodificação e retorna os pesos de atenção.
    - token: tensor (B,1)
    - h: estado oculto do decoder (1,B,H)
    - encoder_outputs: (B,S,H)
    Retorna:
      next_token: (B,1)
      h: (1,B,H)
      attn_weights: (B, 1, S)
    """
    # Acessa os componentes internos do decoder para capturar os pesos de atenção
    embedded_token = decoder.embed(token)  # (B, 1, E)
    gru_output, next_h = decoder.gru(embedded_token, h) # gru_output: (B,1,H), next_h: (1,B,H)

    context, attn_weights = decoder.attn(gru_output, encoder_outputs) # context: (B,1,H), attn_weights: (B,1,S)

    combined = torch.cat([gru_output, context], dim=-1) # (B,1,2H)
    logits = decoder.fc(combined) # (B,1,V)

    next_token = logits[:, -1, :].argmax(-1, keepdim=True) # (B,1)

    return next_token, next_h, attn_weights

def predict_with_attention(model, seq):
    model.eval()
    with torch.no_grad():
        # Codifica e preenche a sequência de entrada
        src_encoded_padded = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)
        src_len_padded = src_encoded_padded.size(1)

        # Passa pelo encoder
        encoder_outputs, h = model.encoder(src_encoded_padded) # encoder_outputs: (B,S,H), h: (1,B,H)

        # Token inicial para o decoder (espaço, <sos>)
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)

        predicted_seq_tokens = []
        attention_weights_list = [] # Lista para armazenar tensores (1, S)

        # Decodifica passo a passo
        for _ in range(max_len): # Usa o max_len global (5) para alinhar com o comprimento da sequência
            token, h, attn_w = decode_step_with_attention(model.decoder, token, h, encoder_outputs)
            predicted_seq_tokens.append(token.item())
            attention_weights_list.append(attn_w.squeeze(0)) # Remove a dimensão do batch, resulta em (1, S)

        predicted_text = decode(predicted_seq_tokens)
        # Concatena todos os pesos de atenção em uma única matriz (Target_Len, Source_Len)
        attention_matrix = torch.cat(attention_weights_list, dim=0)

        return predicted_text, attention_matrix, seq # Retorna a sequência original para os rótulos

# --- Teste e Plotagem ---

# Gerar uma sequência arbitrária
s_input = random_seq(n=5) # n=5 para corresponder ao max_len global
predicted_output, attn_matrix, original_seq = predict_with_attention(model, s_input)

print(f"Sequência de Entrada: '{original_seq}'")
print(f"Sequência Invertida (Real): '{original_seq[::-1]}'")
print(f"Sequência Predita: '{predicted_output}'")

# Preparar rótulos para o plot
# Rótulos da fonte (encoder) - caracteres da sequência original
source_labels = list(original_seq) + [' '] * (max_len - len(original_seq)) # Adiciona padding space se a original for menor

# Rótulos do alvo (decoder) - caracteres da sequência predita
target_labels = list(predicted_output)

plt.figure(figsize=(8, 7))
sns.heatmap(attn_matrix.cpu().numpy(),
            xticklabels=source_labels,
            yticklabels=target_labels,
            cmap='viridis',
            annot=True, fmt=".2f", linewidths=.5)

plt.xlabel('Input Sequence (Encoder)')
plt.ylabel('Output Sequence (Decoder)')
plt.title(f'Attention Weights for Input: "{original_seq}" (Predicted: "{predicted_output}")')
plt.tight_layout()
plt.show()